# CTKM Extractor — chạy bản **Python** trên Google Colab

Đọc ảnh hoặc PDF chứa bảng chương trình khuyến mại (CTKM) và trích xuất ra JSON
11 field.

| Mức | Nội dung | Thời gian |
| --- | --- | --- |
| **1** | Cài + chạy bằng engine fallback `tesseract` | ~2 phút |
| **2** | Bật engine mặc định `paddle_vietocr` (PP-OCR detect + VietOCR) | ~5 phút |
| **3** | Hồ sơ **PDF nhiều trang** | ~1 phút |

Muốn chạy bản **C++** thì dùng [`colab_cpp.ipynb`](colab_cpp.ipynb).

> **Phiên Colab là tạm**: ngắt kết nối là mất sạch `/content`.

---
# Mức 1 — chạy bằng Tesseract

## Cell 1 — lấy code

In [ ]:
!git clone -q https://github.com/Ngoc-LM/OCR_Extractor.git /content/OCR_Extractor
%cd /content/OCR_Extractor
!git log --oneline -3

## Cell 2 — Tesseract + gói tiếng Việt

`tesseract-ocr-vie` là **bắt buộc**: thiếu gói tiếng Việt thì OCR mất sạch dấu.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-vie
!tesseract --list-langs

## Cell 3 — dependency Python

Colab đã có sẵn `cv2`, `numpy`, `PIL`. Chỉ cần thêm mấy gói nhẹ.

In [ ]:
!pip install -q PyYAML pytesseract pytest pymupdf

## Cell 4 — đưa ảnh hoặc PDF vào

In [ ]:
import shlex
from google.colab import files

uploaded = files.upload()          # PNG/JPG hoặc PDF đều được
source = "/content/" + list(uploaded)[0]

# Tên file hay có DẤU CÁCH và dấu tiếng Việt. IPython thay biến vào lệnh shell
# nhưng KHÔNG tự bọc nháy, nên phải tự bọc - nếu không shell tách tên file thành
# nhiều tham số và argparse báo "unrecognized arguments".
src = shlex.quote(source)
print("Đã nhận:", source)
print("Dùng trong lệnh shell:", src)

## Cell 5 — chạy

CLI tự nhận biết đuôi `.pdf` và chuyển sang đường xử lý nhiều trang.

`--debug` in raw OCR text, bảng đã dựng và nguồn của từng field ra stderr — đây là
chỗ để soi khi kết quả sai.

Thêm `--no-binarize` nếu ảnh có **watermark**: adaptive threshold biến nét
watermark mờ thành nét đen đặc đè lên chữ. Đo trên biểu mẫu BM.12 thật, cờ này
đưa bản Python từ 8/11 lên 10/11 field đúng — **nhưng chỉ có lợi với Tesseract**,
với engine mặc định ở Mức 2 thì giữ nguyên mặc định.

In [ ]:
%cd /content/OCR_Extractor/python
!python -m ctkm_extractor.cli --image {src} --out /content/result_tesseract.json \
    --engine tesseract --no-binarize --debug

import json, pathlib

out = pathlib.Path("/content/result_tesseract.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


## Cell 6 — chạy test

In [ ]:
%cd /content/OCR_Extractor/python
!python -m pytest ctkm_extractor/tests -q

---
# Mức 2 — engine mặc định `paddle_vietocr`

Detect bằng PP-OCR (thuật toán DB), recognize bằng **VietOCR** `vgg_transformer` —
recognizer đa ngôn ngữ mặc định của PaddleOCR sai dấu tiếng Việt rõ rệt.

## Cell 7 — cài đặt

**Bắt buộc `--no-deps` cho `vietocr`.** Gói này ghim cứng `pillow==10.2.0` còn
Colab đã có Pillow 11+; cài bình thường sẽ hạ cấp Pillow **ngay dưới chân kernel
đang chạy** và cell sau vỡ với:

```
ImportError: cannot import name 'is_directory' from 'PIL._util'
```

Chuỗi import của `Predictor` thật ra chỉ cần `torch`, `torchvision`, `PIL`,
`numpy`, `einops`, `gdown`, `requests`, `yaml`, `tqdm`. Những gói còn lại mà
`vietocr` khai báo chỉ dùng cho phần **huấn luyện**.

In [ ]:
!pip install -q paddlepaddle paddleocr
!pip install -q --no-deps vietocr
!pip install -q einops gdown

import PIL
from PIL import ImageFont          # chỗ vỡ nếu Pillow bị hạ cấp
print("Pillow", PIL.__version__, "- OK")

⚠️ **Sau cell trên nên `Runtime → Restart session`** rồi chạy tiếp từ Cell 8:
`paddleocr` có thể thay `numpy`/`opencv` mà Colab đã nạp sẵn vào bộ nhớ.

## Cell 8 — chạy

`--strict-engine` **cấm fallback**: thấy dòng `Dùng OCR provider 'paddle_vietocr'`
là chắc chắn đang chạy đúng engine, thiếu gì nó báo lỗi thay vì âm thầm chuyển
sang Tesseract.

Lần chạy đầu tải model về cache (~1 phút): text detector của PP-OCR và weights
VietOCR `vgg_transformer` (~151 MB).

In [ ]:
%cd /content/OCR_Extractor/python
!python -m ctkm_extractor.cli --image {src} --out /content/result_vietocr.json \
    --strict-engine --debug

import json, pathlib

out = pathlib.Path("/content/result_vietocr.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


### Hai điều đã biết về đường Paddle trên Colab

**oneDNN mặc định TẮT.** Tổ hợp `paddlepaddle` mới (IR = PIR) + oneDNN chết ngay
lúc inference với `ConvertPirAttribute2RuntimeAttribute not support ...`. Muốn
chạy nhanh hơn thì bật lại; backend lỗi thì provider tự tắt và thử lại một lần:

```python
CTKMExtractor(engine="paddle_vietocr", provider_kwargs={"enable_mkldnn": True})
```

**PaddleX chỉ khởi tạo được MỘT lần mỗi process.** Dựng provider thứ hai trong
cùng kernel sẽ gặp `PDX has already been initialized`. Provider dùng lại detector
đã có nên vòng lặp trong notebook vẫn chạy, nhưng muốn thật sự đổi tham số
detector thì phải chạy tiến trình mới (mỗi lệnh `!python -m ...` là một tiến
trình riêng nên không vướng).

---
# Mức 3 — hồ sơ PDF nhiều trang

Bảng CTKM thường chỉ nằm ở **một hoặc vài trang** trong cả tập hồ sơ, nên pipeline
không đoán trang nào chứa bảng mà **chấm điểm từng trang**:

1. Điểm mỗi trang = *(số field trích được, điểm khớp trung bình)*. Trang không có
   bảng ra `(0, 0.0)` nên không bao giờ được chọn khi có trang khác.
2. **Trang chính** = trang điểm cao nhất; hoà thì lấy trang số nhỏ hơn.
3. Field nào **vẫn thiếu** mới lấy bù từ trang khác — cho trường hợp bảng bị tách
   qua nhiều trang. Bước này **không ghi đè** giá trị của trang chính, vì trang
   chính là trang thật sự chứa bảng còn trang khác dễ có nhãn trùng tên nằm trong
   phần văn bản thường.

## Cell 9 — chạy trên PDF

In [ ]:
%cd /content/OCR_Extractor/python
!python -m ctkm_extractor.cli --pdf {src} --out /content/result_pdf.json \
    --strict-engine --debug

import json, pathlib

out = pathlib.Path("/content/result_pdf.json")
if not out.is_file():
    raise SystemExit("Lệnh trên THẤT BẠI - xem log phía trên, chưa có file kết quả.")
print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


Log sẽ cho biết trang nào có dữ liệu và trang nào được chọn:

```
INFO extractor: Trang 1: trích được 0 field
INFO extractor: Trang 2: trích được 9 field
INFO extractor: Trang 3: trích được 0 field
INFO extractor: Trang không có dữ liệu CTKM: 1, 3
INFO extractor: Chọn trang 2 làm trang chính trong 3 trang

Trang            : 2 (chính) trong 3 trang đã OCR
```

Field lấy bù từ trang khác có `trang=N` ở cuối dòng trong mục *Nguồn từng field*.

**Biết trước trang nào** thì chỉ định để chạy nhanh hơn nhiều:

```bash
!python -m ctkm_extractor.cli --pdf {src} --pages 2-3 --out /content/r.json
```

Các cờ liên quan: `--pages "1,3-5"`, `--dpi 300` (mặc định, mức đã kiểm chứng),
`--keep-pages` (giữ ảnh từng trang đã render để soi).

---
## Dùng như thư viện

```python
%cd /content/OCR_Extractor/python
from ctkm_extractor.extraction.extractor import CTKMExtractor

extractor = CTKMExtractor(engine="paddle_vietocr", strict_engine=True)
result = extractor.extract_from_pdf("/content/ho_so.pdf", dpi=300, pages="2")
print(result.to_json())
print(result.debug_report())          # bảng đã dựng + nguồn từng field
```

## Lưu kết quả sang Drive

```python
from google.colab import drive
drive.mount("/content/drive")
!cp /content/result_*.json /content/drive/MyDrive/
```

---
## Sự cố hay gặp

| Triệu chứng | Nguyên nhân / cách xử lý |
| --- | --- |
| `ImportError: cannot import name 'is_directory' from 'PIL._util'` | `vietocr` đã hạ cấp Pillow. Cài lại bằng `--no-deps` như Cell 7, rồi **Restart session** |
| `error: unrecognized arguments: ...` | Tên file có **dấu cách**. Notebook đã bọc nháy bằng `shlex.quote`; nếu tự gõ lệnh thì nhớ `--pdf "tên file.pdf"` |
| Chữ tiếng Việt mất dấu | Thiếu `tesseract-ocr-vie`. Kiểm tra bằng `!tesseract --list-langs` |
| `PDX has already been initialized` | PaddleX chỉ khởi tạo một lần mỗi process. Restart session, hoặc chạy mỗi cấu hình bằng một lệnh `!python -m ...` riêng |
| `ConvertPirAttribute2RuntimeAttribute not support` | Backend oneDNN của paddle. Provider tự tắt oneDNN và thử lại; nếu vẫn lỗi thì dùng `--engine tesseract` |
| `Cần pymupdf để đọc PDF` | `!pip install pymupdf` |
| Kết quả toàn `null` | Xem `--debug`, mục *OCR raw text* trước: text thô đã sai thì vấn đề ở ảnh/OCR chứ không phải tầng trích xuất |
| Trang chính bị chọn sai | Xem log `Trang N: trích được K field`. Nếu biết trước thì ép bằng `--pages` |
| Field đúng ra phải có nhưng lại `null` | Xem mục *Cảnh báo*: có thể collision guard đã loại vì hai field cùng khớp một ô bảng |